# Live League Games

In [1]:
%load_ext autoreload
%autoreload 2
import requests
from dotenv import load_dotenv
import os
import pandas as pd
from datetime import datetime as dt
import numpy as np
import json
from retry import retry 
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)
from src.config import ROOT_DIR

In [2]:
# Steam API constants
STEAM_URL = 'http://api.steampowered.com/'
LIVE_LEAGUE_GAMES = 'IDOTA2Match_570/GetLiveLeagueGames/v1'
REAL_TIME_STATS = 'IDOTA2MatchStats_570/GetRealtimeStats/v1' # Requires server_steam_id 

load_dotenv()
API_KEY = os.getenv('STEAM_API')

### Fetching premium and professional leagues' match details

In [3]:
# Function to retrieve Json from Steam WebAPI
session = requests.Session()
session.params.update({'key': API_KEY})

@retry(tries=3, delay=2)
def fetch_live_league_games():
    try:
        url = f'{STEAM_URL}{LIVE_LEAGUE_GAMES}'
        res = session.get(url)
        match_details = res.json()
        if not match_details:
            raise ValueError("Empty dictionary, retrying...")
        else:
            return match_details
    except Exception as err:
        print("Did not get a response, retrying...")
        raise

In [4]:
game_data = fetch_live_league_games()
games = game_data['result']['games']

In [10]:
games[0]

{'players': [{'account_id': 1035926100,
   'name': '777',
   'hero_id': 0,
   'team': 2},
  {'account_id': 1220632105, 'name': 'Norbert', 'hero_id': 5, 'team': 1},
  {'account_id': 1749799475, 'name': 'LV.Noxen', 'hero_id': 32, 'team': 0},
  {'account_id': 1083977859, 'name': 'VittoBossG', 'hero_id': 72, 'team': 1},
  {'account_id': 151588437,
   'name': 'Sebastian Pereyra',
   'hero_id': 0,
   'team': 4},
  {'account_id': 1122407405, 'name': 'Bandana', 'hero_id': 38, 'team': 1},
  {'account_id': 1750299292, 'name': 'LV.Duskworn', 'hero_id': 74, 'team': 0},
  {'account_id': 127731869, 'name': 'samosval301', 'hero_id': 0, 'team': 4},
  {'account_id': 1750300895, 'name': 'teresaross', 'hero_id': 68, 'team': 0},
  {'account_id': 1750623235, 'name': 'LV.Velm', 'hero_id': 10, 'team': 0},
  {'account_id': 1750510331, 'name': 'LV.Ashreaver', 'hero_id': 55, 'team': 0},
  {'account_id': 1119908658, 'name': 'TwoFaced', 'hero_id': 39, 'team': 1},
  {'account_id': 1216609165, 'name': 'Satoru', 'he

In [13]:
from src.pydantic_models.live_league_games import LiveLeagueGame
LiveLeagueGame(**games[0])

LiveLeagueGame(match_id=8270782741, league_id=17233, start_time=1745750479.916811, radiant_team=TeamData(team_name='Lunar Vibes', team_id=9749662), dire_team=TeamData(team_name='Bright Crusaders', team_id=8629317), scoreboard=ScoreBoard(duration=343.8333740234375, radiant=Faction(players=[Player(player_slot=0, account_id=1750623235, hero_id=10), Player(player_slot=1, account_id=1750510331, hero_id=55), Player(player_slot=2, account_id=1750299292, hero_id=74), Player(player_slot=3, account_id=1750300895, hero_id=68), Player(player_slot=4, account_id=1749799475, hero_id=32)]), dire=Faction(players=[Player(player_slot=128, account_id=1122407405, hero_id=38), Player(player_slot=129, account_id=1220632105, hero_id=5), Player(player_slot=130, account_id=1119908658, hero_id=39), Player(player_slot=131, account_id=1216609165, hero_id=105), Player(player_slot=132, account_id=1083977859, hero_id=72)])))

In [24]:
scoreboard = games[0]['scoreboard']
scoreboard

{'duration': 1203.066650390625,
 'roshan_respawn_timer': 0,
 'radiant': {'score': 10,
  'tower_state': 1974,
  'barracks_state': 63,
  'picks': [{'hero_id': 68},
   {'hero_id': 102},
   {'hero_id': 35},
   {'hero_id': 29},
   {'hero_id': 4}],
  'bans': [{'hero_id': 19},
   {'hero_id': 38},
   {'hero_id': 60},
   {'hero_id': 129},
   {'hero_id': 106},
   {'hero_id': 128},
   {'hero_id': 86}],
  'players': [{'player_slot': 0,
    'account_id': 19732051,
    'hero_id': 35,
    'kills': 4,
    'death': 2,
    'assists': 4,
    'last_hits': 135,
    'denies': 21,
    'gold': 4,
    'level': 14,
    'gold_per_min': 455,
    'xp_per_min': 564,
    'ultimate_state': 3,
    'ultimate_cooldown': 0,
    'item0': 75,
    'item1': 172,
    'item2': 75,
    'item3': 63,
    'item4': 166,
    'item5': 265,
    'respawn_timer': 0,
    'position_x': -2371.248046875,
    'position_y': -3098.685546875,
    'net_worth': 8889},
   {'player_slot': 1,
    'account_id': 99260250,
    'hero_id': 4,
    'kills'

In [34]:
scoreboard.keys()
scoreboard['radiant']
scoreboard['radiant'].keys()
scoreboard['radiant']['players'][0]

{'player_slot': 0,
 'account_id': 19732051,
 'hero_id': 35,
 'kills': 4,
 'death': 2,
 'assists': 4,
 'last_hits': 135,
 'denies': 21,
 'gold': 4,
 'level': 14,
 'gold_per_min': 455,
 'xp_per_min': 564,
 'ultimate_state': 3,
 'ultimate_cooldown': 0,
 'item0': 75,
 'item1': 172,
 'item2': 75,
 'item3': 63,
 'item4': 166,
 'item5': 265,
 'respawn_timer': 0,
 'position_x': -2371.248046875,
 'position_y': -3098.685546875,
 'net_worth': 8889}

In [36]:
type(scoreboard['radiant']['players'][0]['account_id'])
type(scoreboard['radiant']['players'][0]['hero_id'])

int

In [8]:
json.loads(match.to_json(orient='records', date_format='iso'))

[{'players': [{'account_id': 938076668,
    'name': 'Spell',
    'hero_id': 0,
    'team': 4},
   {'account_id': 860271515, 'name': 'null', 'hero_id': 53, 'team': 1},
   {'account_id': 911479785, 'name': 'ttQ', 'hero_id': 100, 'team': 1},
   {'account_id': 78392868, 'name': 'Real_Good', 'hero_id': 39, 'team': 1},
   {'account_id': 149014813, 'name': 'Artful-', 'hero_id': 135, 'team': 1},
   {'account_id': 358426134, 'name': 'HAOS', 'hero_id': 138, 'team': 1},
   {'account_id': 1073759713, 'name': 'чижик', 'hero_id': 22, 'team': 0},
   {'account_id': 130169737, 'name': 'uselesscloud', 'hero_id': 64, 'team': 0},
   {'account_id': 141557630, 'name': 'hotaken', 'hero_id': 6, 'team': 0},
   {'account_id': 129231690, 'name': 'Paranoia', 'hero_id': 29, 'team': 0},
   {'account_id': 129869699, 'name': 'St_Ilia', 'hero_id': 87, 'team': 0}],
  'radiant_team': {'team_name': 'aut_chayhana',
   'team_id': 9735655,
   'team_logo': 38950614858552719,
   'complete': True},
  'dire_team': {'team_name':

In [6]:
# Import the list of premium and professional league games id

import yaml

file_path = os.path.join(ROOT_DIR, f'constants/league_ids.yml')

with open(file_path, 'r') as file:
    content = yaml.safe_load(file) or {}
    if 'PREMIUM_LEAGUES' in content:
        premium_leagues = content['PREMIUM_LEAGUES']
    if 'PROFESSIONAL_LEAGUES' in content:
        professional_leagues = content['PROFESSIONAL_LEAGUES']
        
premium_list = list(premium_leagues.values())
professional_list = list(professional_leagues.values())



In [7]:
from src.pydantic_models.match import Match
from src.pydantic_models.live_league_games import LiveLeagueGames

In [8]:
live_league_games = []

for row in games:
    
    league_id = row.get('league_id', np.nan)
    if league_id in premium_list + professional_list:
    
        game_data = LiveLeagueGames(**row)
        
        # Populate common fields
        match_data = {
            'match_id': game_data.match_id,
            'radiant_team_id': game_data.radiant_team.team_id,
            'radiant_name': game_data.radiant_team.team_name,
            'dire_team_id': game_data.dire_team.team_id,
            'dire_name': game_data.dire_team.team_name,
            'duration': game_data.scoreboard.duration,
            'start_time': int(dt.now().timestamp())
        }
        
        # Populate player data
        for team in ['radiant', 'dire']:
            faction = getattr(game_data.scoreboard, team)
            for player in faction.players:
                slot = player.player_slot
                player_data = {
                    f"slot_{slot}_account_id": player.account_id,
                    f"slot_{slot}_hero_id": player.hero_id
                } 
                match_data.update(player_data)
                
        live_league_games.append(Match(**match_data))
                

    
if len(live_league_games) == 0:
    print("No premium or professional games right now")
else:
    print(len(live_league_games))
    print(live_league_games)   


No premium or professional games right now


In [9]:
live_league_games

[]